# LoRA Fine-Tuning and Evaluation of LLaMA 3.2 for Math Reasoning

This notebook demonstrates how to fine-tune the LLaMA 3.2 (1B) language model using LoRA (Parameter-Efficient Fine-Tuning) on a math word-problem dataset. The workflow covers loading the base model, attaching LoRA adapters, preprocessing and splitting the dataset into training and evaluation sets, configuring a training loop with periodic evaluation, and saving the resulting LoRA adapters. The notebook also includes inference examples to qualitatively validate the fine-tuned model.Dans ce notebook, j’ai mis en place un pipeline complet pour affiner le modèle LLaMA 3.2 en utilisant la technique LoRA, qui permet d’entraîner un modèle de manière efficace sans modifier tous ses paramètres.Le travail consiste à charger le modèle de base, ajouter des adaptateurs LoRA, préparer un jeu de données de problèmes mathématiques, séparer les données en entraînement et évaluation

In [1]:
!pip install transformers datasets accelerate peft

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com

[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip


## Imports & environment checks
Imports all libraries needed for training

Verifies CUDA is available

Confirms which GPU is being used

In [2]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model


In [3]:
assert torch.cuda.is_available(), "GPU required"
print(torch.cuda.get_device_name(0))


NVIDIA H100 NVL


## Load tokenizer & base model (LLaMA 3.2)
this cell initializes the base (pre-trained) language model and its tokenizer, which are required before applying LoRA fine-tuning or running inference.

Tokenizer
The tokenizer converts raw text into token IDs that the model can process, and converts generated token IDs back into text.
We explicitly set the pad_token to the eos_token because LLaMA models do not define a padding token by default. This is necessary for batching and open-ended text generation.

Model
The model is loaded from Hugging Face using the AutoModelForCausalLM interface, which automatically selects the correct architecture for LLaMA.
The model is loaded:

In bfloat16 precision to reduce memory usage and improve performance on modern GPUs

With device_map="auto" so that Hugging Face automatically places the model on the available GPU(s)

In [4]:
HF_MODEL_ID = "meta-llama/Llama-3.2-1B"

tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token


tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

In [5]:
model = AutoModelForCausalLM.from_pretrained(
    HF_MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

print("Base model loaded successfully")



config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

Base model loaded successfully


## Attach LoRA adapters (PEFT)
This cell applies LoRA (Low-Rank Adaptation) adapters to the base LLaMA model in order to fine-tune it efficiently without updating all model parameters.

LoRA configuration
The LoraConfig defines how the adapters are injected:

r=16 specifies the rank of the low-rank update matrices (higher values increase capacity but also memory usage)

lora_alpha=32 controls the scaling of the LoRA updates

lora_dropout=0.05 adds regularization to prevent overfitting

target_modules selects the attention projection layers (q_proj, k_proj, v_proj, o_proj) where LoRA is applied, which is the standard and most effective choice for LLaMA models

task_type="CAUSAL_LM" indicates that the model is being fine-tuned for causal language modeling

Adapter injection
get_peft_model wraps the base model and inserts the LoRA adapters while freezing the original model weights. During training, only the LoRA parameters are updated.

Trainable parameter summary
print_trainable_parameters() reports how many parameters are trainable versus frozen.
This confirms that less than 1% of the total parameters are being trained, dramatically reducing memory usage

In [6]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 3,407,872 || all params: 1,239,222,272 || trainable%: 0.2750


## Load dataset

In [7]:
raw_dataset = load_dataset(
    "microsoft/orca-math-word-problems-200k",
    split="train"
)


README.md: 0.00B [00:00, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/84.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/200035 [00:00<?, ? examples/s]

## Create train / evaluation split

In [8]:
split_dataset = raw_dataset.train_test_split(
    test_size=0.05,
    seed=42
)

train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print(len(train_dataset), len(eval_dataset))


190033 10002


## Reduce size for quick testing

In [9]:
train_dataset = train_dataset.select(range(100))
eval_dataset = eval_dataset.select(range(50))


## Tokenization & formatting
This cell prepares the raw text data for training and evaluation by converting it into a numerical format that the language model can process.

Maximum sequence length
MAX_SEQ_LEN = 1024 defines the maximum number of tokens per example.
Preprocessing function
The preprocess function applies the tokenizer to each dataset example:

It tokenizes the question field from the dataset.Label construction for causal language modeling
For causal language modeling, the target output is the next token in the same sequence.
Applying preprocessing to datasets
The preprocessing function is applied to both the training and evaluation datasets using Dataset.map():Therefore, the labels field is created by copying input_ids.

In [10]:
MAX_SEQ_LEN = 1024

def preprocess(example):
    tokens = tokenizer(
        example["question"],
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding="max_length",
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens


In [11]:
train_dataset = train_dataset.map(
    preprocess,
    remove_columns=train_dataset.column_names,
    num_proc=4,
)

eval_dataset = eval_dataset.map(
    preprocess,
    remove_columns=eval_dataset.column_names,
    num_proc=4,
)


Map (num_proc=4):   0%|          | 0/100 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/50 [00:00<?, ? examples/s]

## Training configuration (WITH evaluation)
This cell defines how the model is trained and evaluated. It configures the training process but does not start it.
The model is trained with a small per-device batch size and gradient accumulation to simulate a larger effective batch size while staying within GPU memory limits.
A learning rate suitable for LoRA fine-tuning is used, and the model is trained for a fixed number of epochs.

Training loss is logged regularly to monitor convergence.
Evaluation is automatically run every fixed number of steps on a separate evaluation dataset. This evaluation does not update model weights.
Model checkpoints are saved periodically, with a limit on how many are kept.
bfloat16 precision, a cosine learning-rate scheduler, and warmup steps are used to improve training stability and efficiency.

External experiment tracking tools are disabled for simplicity.

This configuration provides a stable and efficient training setup with built-in evaluation.

In [12]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./checkpoints",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=32,   # global batch = 32
    learning_rate=2e-4,
    num_train_epochs=1,

    logging_steps=20,

    evaluation_strategy="steps",
    eval_steps=50,                    # evaluate every 50 steps

    save_steps=2000,
    save_total_limit=2,

    bf16=True,
    optim="adamw_torch",
    lr_scheduler_type="cosine",
    warmup_steps=500,
    report_to="none",
)


/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1545: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


## Trainer (training + evaluation engine)
This cell creates the Trainer, which is the component responsible for running both training and evaluation.
The data collator prepares batches of tokenized data for causal language modeling (mlm=False), ensuring inputs and labels are correctly aligned.

The Trainer is initialized with:
the model (with LoRA adapters attached),
the training configuration (training_args),
the training dataset,
the evaluation dataset.

During training, the Trainer:
updates model weights using the training dataset,
periodically runs evaluation on the evaluation dataset,
logs metrics and saves checkpoints according to the configuration.

This setup centralizes the entire training and evaluation workflow in a single, reusable engine.

In [13]:
from transformers import Trainer, DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

## Start training (with evaluation)

In [14]:
trainer.train()


Step,Training Loss,Validation Loss


TrainOutput(global_step=3, training_loss=2.1867807706197104, metrics={'train_runtime': 5.9757, 'train_samples_per_second': 16.734, 'train_steps_per_second': 0.502, 'total_flos': 575994977058816.0, 'train_loss': 2.1867807706197104, 'epoch': 0.96})

## Manual evaluation

In [15]:
metrics = trainer.evaluate()
print(metrics)


{'eval_loss': 2.1743762493133545, 'eval_runtime': 0.6915, 'eval_samples_per_second': 72.311, 'eval_steps_per_second': 10.124, 'epoch': 0.96}


## Save LoRA adapters ONLY
This cell saves only the LoRA adapter weights, not the full base model.
The base LLaMA model remains unchanged and is not duplicated, which keeps the saved files small.This approach provides an efficient and portable way to store and reuse fine-tuned LoRA weights.

In [16]:
model.save_pretrained("./lora_llama3_math")
tokenizer.save_pretrained("./lora_llama3_math")

print("LoRA adapters saved")

LoRA adapters saved


## Test inference

In [17]:
model.eval()

prompt = (
    "Solve the problem and give ONLY the final numeric answer.\n\n"
    "If a rectangle has length 5 and width 3, what is its area?\n\n"
    "Answer:"
)

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))



/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:601: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:606: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)


Solve the problem and give ONLY the final numeric answer.

If a rectangle has length 5 and width 3, what is its area?

Answer: 15

Explanation: The area of a rectangle is length times width. The length is 5
